# 01B · TCGA COAD (cBioPortal PanCanAtlas 2018) — Intake, QC, and DEGs (Beginner-Friendly)

**Owner:** Pranali

## Goal
From cBioPortal RSEM expression + clinical sample file, produce:
- `data_processed/tcga/tcga_log2_rsem.csv`
- `data_processed/tcga/tcga_clinical_sample_clean.csv`
- `results/de/tcga/TCGA_DEG_Tumor_vs_Normal.csv`

## Files required in Drive
Put these inside:
`{BASE}/data_raw/tcga_cbioportal/`

- `data_mrna_seq_v2_rsem.txt`
- `data_clinical_sample.txt`

### Mount + paths

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

BASE = "/content/drive/MyDrive/GeneticCodonShared/Hippo_Dysbiosis" # change accordingly

RAW_EXPR  = f"{BASE}/data_raw/tcga_cbioportal/data_mrna_seq_v2_rsem.txt"
RAW_CLSMP = f"{BASE}/data_raw/tcga_cbioportal/data_clinical_sample.txt"

OUT_PROC = f"{BASE}/data_processed/tcga"
OUT_DE   = f"{BASE}/results/de/tcga"
OUT_FIG  = f"{BASE}/results/figures/tcga"

import os
os.makedirs(OUT_PROC, exist_ok=True)
os.makedirs(OUT_DE,   exist_ok=True)
os.makedirs(OUT_FIG,  exist_ok=True)

print("EXPR:", RAW_EXPR)
print("CLIN:", RAW_CLSMP)


### Imports

In [ ]:
import pandas as pd, numpy as np, re
from scipy.stats import ttest_ind
from statsmodels.stats.multitest import multipletests
import matplotlib.pyplot as plt
import seaborn as sns

### Load expression data

In [ ]:
expr = pd.read_csv(RAW_EXPR, sep="\t", comment="#")
print("Raw expr shape:", expr.shape)
display(expr.head())

# Detect gene column
gene_col = None
for c in expr.columns[:5]:
    if str(c).lower() in ["hugo_symbol","gene","symbol","gene_symbol"]:
        gene_col = c
        break
if gene_col is None:
    gene_col = expr.columns[0]
print("Using gene column:", gene_col)

expr = expr.dropna(subset=[gene_col])
expr[gene_col] = expr[gene_col].astype(str).str.strip()
expr = expr.set_index(gene_col)

# Drop Entrez column if present
for drop_c in ["Entrez_Gene_Id","entrez_gene_id","Entrez Gene Id"]:
    if drop_c in expr.columns:
        expr = expr.drop(columns=[drop_c])

expr = expr.apply(pd.to_numeric, errors="coerce").fillna(0)

dup = int(expr.index.duplicated().sum())
print("Duplicate genes:", dup)
if dup:
    expr = expr.groupby(expr.index).mean()

print("Clean expr shape:", expr.shape)


### Load clinical sample + infer Tumor/Normal

In [ ]:
clin = pd.read_csv(RAW_CLSMP, sep="\t", comment="#")
print("Clinical shape:", clin.shape)
display(clin.head())

cand_id = [c for c in clin.columns if re.search("sample|barcode|submitter", c, flags=re.I)]
sample_id_col = cand_id[0] if cand_id else clin.columns[0]
print("Sample ID col:", sample_id_col)
clin[sample_id_col] = clin[sample_id_col].astype(str).str.strip()

cand_type = [c for c in clin.columns if re.search("sample_type|tumor|normal", c, flags=re.I)]
if not cand_type:
    raise ValueError("Could not detect tumor/normal column. Open the file and pick the correct column name.")
type_col = cand_type[0]
print("Type col:", type_col)

clin[type_col] = clin[type_col].astype(str).str.lower()
clin["group"] = np.where(clin[type_col].str.contains("normal"), "Normal", "Tumor")

print("Group counts:")
display(clin["group"].value_counts())


### Align expression with clinical

In [ ]:
expr_cols = pd.Index(expr.columns.astype(str).str.strip())
clin_ids  = pd.Index(clin[sample_id_col].astype(str).str.strip())

overlap = expr_cols.intersection(clin_ids)
print("Overlap samples:", len(overlap))

expr = expr.loc[:, overlap]
clin = clin[clin[sample_id_col].isin(overlap)].copy()
clin = clin.set_index(sample_id_col).loc[overlap].reset_index()

print("Final expr:", expr.shape)
print("Final clin:", clin.shape)

clin.to_csv(f"{OUT_PROC}/tcga_clinical_sample_clean.csv", index=False)
print("Saved ->", f"{OUT_PROC}/tcga_clinical_sample_clean.csv")


### Log2 Transform

In [ ]:
expr_log = np.log2(expr + 1)
expr_log.to_csv(f"{OUT_PROC}/tcga_log2_rsem.csv")
print("Saved ->", f"{OUT_PROC}/tcga_log2_rsem.csv")


### DEG Tumor vs Normal

In [ ]:
groups = clin["group"].astype(str).values
i1 = np.where(groups=="Tumor")[0]
i2 = np.where(groups=="Normal")[0]
print(f"Group sizes: Tumor={len(i1)}, Normal={len(i2)}")

X1 = expr_log.iloc[:, i1]
X2 = expr_log.iloc[:, i2]

mean1, mean2 = X1.mean(1), X2.mean(1)
logFC = mean1 - mean2

pvals = np.array([
    ttest_ind(X1.iloc[i,:], X2.iloc[i,:], equal_var=False).pvalue
    for i in range(expr_log.shape[0])
])
padj = multipletests(pvals, method="fdr_bh")[1]

deg = pd.DataFrame({
    "gene": expr_log.index.astype(str),
    "mean_Tumor": mean1.values,
    "mean_Normal": mean2.values,
    "logFC_Tumor_minus_Normal": logFC.values,
    "pval": pvals,
    "padj_fdr": padj
}).sort_values("padj_fdr")

display(deg.head(10))
deg.to_csv(f"{OUT_DE}/TCGA_DEG_Tumor_vs_Normal.csv", index=False)
print("Saved ->", f"{OUT_DE}/TCGA_DEG_Tumor_vs_Normal.csv")


### DEG analysis

In [ ]:
df = deg.copy()
df["neglog10_fdr"] = -np.log10(df["padj_fdr"] + 1e-300)

plt.figure(figsize=(6,4))
plt.scatter(df["logFC_Tumor_minus_Normal"], df["neglog10_fdr"], alpha=0.35)
plt.axvline(1, linestyle="--"); plt.axvline(-1, linestyle="--")
plt.axhline(-np.log10(0.05), linestyle="--")
plt.title("TCGA COAD Volcano: Tumor vs Normal")
plt.xlabel("log2FC"); plt.ylabel("-log10(FDR)")
plt.tight_layout()
plt.savefig(f"{OUT_FIG}/volcano_Tumor_vs_Normal.png", dpi=200)
plt.show()
